# 07. 나만의 분류기 — 특징 뽑기 + numpy 로 직접 학습

1. **특징 뽑기**: 이미 배운 모델(MobileNetV2)의 마지막 층 앞에서 숫자 1280개를 꺼낸다 — 사진 한 장 = 숫자 1280개.
2. **학습**: 그 숫자들로 "어느 종류인가" 를 맞히는 작은 모델(소프트맥스 회귀)을 **numpy 로 직접** 학습한다.
3. 웹캠으로 종류마다 사진을 모으고, 학습하고, 새 사진으로 맞혀 본다.

edge-lab 의 [가르치기] 가 하는 일과 같다 (거기는 브라우저에서 학습한다).

## 1. 특징 뽑는 모델

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

def pick(*prefer):
    """먼저 적은 장치부터, 이 PC 에 있는 것을 고른다."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame


DEVICE = pick("NPU", "GPU", "CPU")
backbone = core.compile_model(MODELS / "backbone" / "mobilenetv2_feat.xml", DEVICE)
print(DEVICE, "입력:", backbone.input(0).shape, "출력:", backbone.output(0).shape)

def feature(bgr):
    """사진 → 숫자 1280개. 입력은 (1, 224, 224, 3) RGB 0~1 — 가르치기 화면(TF.js)과 같은 전처리."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = cv2.resize(rgb, (224, 224), interpolation=cv2.INTER_NEAREST).astype(np.float32) / 255.0
    return backbone(x[None])[backbone.output(0)].reshape(-1)

## 2. 사진 모으기
`LABELS` 에 종류를 적는다. 종류마다 웹캠이 `N` 장을 조금씩 간격을 두고 찍는다 — 그동안 물건의 각도·거리를 바꿔 준다.
(웹캠이 없으면 `data/<종류>/*.jpg` 폴더에서 읽는다.)

In [ ]:
LABELS = ["가위", "바위", "보"]
N = 20

def capture(n, index=0, gap=0.15):
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    shots = []
    for _ in range(5):
        cap.read()                                    # 처음 몇 장은 버린다
    while len(shots) < n:
        ok, f = cap.read()
        if not ok:
            break
        shots.append(f)
        time.sleep(gap)
    cap.release()
    return shots

X, y = [], []
for k, label in enumerate(LABELS):
    folder = Path("data") / label
    if folder.is_dir():
        shots = [cv2.imread(str(p)) for p in sorted(folder.glob("*.jpg"))]
    else:
        input(f"'{label}' 을(를) 웹캠에 보여 주고 Enter")
        shots = capture(N)
    shots = [s for s in shots if s is not None]
    print(f"{label}: {len(shots)}장")
    for s in shots:
        X.append(feature(s))
        y.append(k)
X, y = np.array(X, np.float32), np.array(y)
print("특징 행렬:", X.shape)

## 3. 소프트맥스 회귀를 직접 학습
- 점수 = `X @ W + b` (종류마다 점수 하나) → 소프트맥스로 확률로 바꾼다
- 틀린 만큼(교차 엔트로피) 줄어드는 쪽으로 `W`, `b` 를 조금씩 고친다 (경사 하강)

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
cut = int(len(X) * 0.8)                               # 80% 로 배우고 20% 로 시험
tr, te = idx[:cut], idx[cut:]

mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6           # 특징마다 크기를 맞춘다
Z = (X - mu) / sd
K, D = len(LABELS), X.shape[1]
W = np.zeros((D, K), np.float32)
b = np.zeros(K, np.float32)

def softmax(s):
    s = s - s.max(1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(1, keepdims=True)

onehot = np.eye(K)[y]
lr, lam = 0.05, 1e-3
for step in range(301):
    p = softmax(Z[tr] @ W + b)
    loss = float(-np.log(np.clip(p[np.arange(len(tr)), y[tr]], 1e-9, 1)).mean()) + 0.0   # + 0.0: -0.0 을 0.0 으로
    g = (p - onehot[tr]) / len(tr)                    # 점수에 대한 기울기
    W -= lr * (Z[tr].T @ g + lam * W)
    b -= lr * g.sum(0)
    if step % 50 == 0:
        acc_tr = (p.argmax(1) == y[tr]).mean()
        acc_te = (softmax(Z[te] @ W + b).argmax(1) == y[te]).mean() if len(te) else float("nan")
        print(f"{step:3d}  손실 {loss:.3f}  배운 것 {acc_tr:.0%}  시험 {acc_te:.0%}")

## 4. 새 사진으로 맞혀 보기

In [ ]:
def predict(bgr):
    p = softmax(((feature(bgr) - mu) / sd)[None] @ W + b)[0]
    return LABELS[int(p.argmax())], p

shot = grab()
label, p = predict(shot)
show(shot, 320)
print("답:", label, "| 확률:", {l: round(float(v), 2) for l, v in zip(LABELS, p)})

## 5. 저장 · 불러오기

In [ ]:
np.savez("my_classifier.npz", W=W, b=b, mu=mu, sd=sd, labels=np.array(LABELS))
m = np.load("my_classifier.npz")
print("저장한 종류:", list(m["labels"]), "| W", m["W"].shape)

## 해 볼 것
1. `N` 을 5 로 줄이면 시험 정확도가 어떻게 되나? 40 으로 늘리면?
2. `lr` 을 0.5 · 0.005 로 바꾸면 손실이 어떻게 줄어드나?
3. 소프트맥스 대신 **가장 가까운 평균**(종류마다 특징 평균을 내고, 새 사진과 가장 가까운 평균의 종류)으로 해 보자. 몇 줄이면 되나?
4. 06 번의 손 점 63개를 특징으로 바꿔 손동작 분류기를 만들어 보자. 배경이 바뀌어도 잘 맞히나?